In [ ]:
# Configuração (células de código NÃO aparecem no PDF; as figuras são guardadas em figures/
# e referenciadas nas células de texto com ![Legenda](figures/x.pdf){#fig:id width=80%})
from pathlib import Path
import matplotlib.pyplot as plt
ROOT = Path.cwd()
FIG, DATA = ROOT / "figures", ROOT / "data"
FIG.mkdir(exist_ok=True)
plt.rcParams.update({"font.family": "sans-serif", "font.size": 9, "savefig.bbox": "tight", "figure.dpi": 150})

# Introdução

A função de uma proteína é determinada, em grande medida, pela sua estrutura tridimensional, que por sua vez está codificada na sequência de aminoácidos \cite{anfinsen1973}. Prever a estrutura a partir da sequência — o chamado *problema do enovelamento proteico* — foi durante cerca de 50 anos um dos grandes desafios em aberto da biologia. Na 14.ª edição do CASP (*Critical Assessment of protein Structure Prediction*), em 2020, o AlphaFold 2, desenvolvido pela Google DeepMind, atingiu pela primeira vez uma exatidão comparável à de métodos experimentais na maioria dos alvos \cite{jumper2021,kryshtafovych2021casp14}. Este avanço valeu a Demis Hassabis e John Jumper o Prémio Nobel da Química de 2024.

Em 2024, o AlphaFold 3 alargou a previsão a complexos com ácidos nucleicos, pequenas moléculas e modificações químicas \cite{abramson2024af3}.

Neste trabalho analisa-se o AlphaFold, nas versões 2 (AF2) e 3 (AF3), segundo os quatro eixos propostos: enquadramento biológico e ómico (Secção 2), arquitetura e implementação computacional (Secção 3), algoritmo e fluxo de trabalho (Secção 4) e reprodutibilidade, comunidade e limitações (Secção 5). Para além da análise documental, usaram-se as três principais vias de acesso à ferramenta para prever duas proteínas humanas, a ubiquitina e a hemoglobina, comparando os resultados com estruturas experimentais, de forma a ilustrar na prática o que o AlphaFold devolve, como se interpretam os seus resultados e quais as suas limitações.

# Enquadramento Biológico e Contexto Ómico

## Ómica(s) abrangida(s)

O AlphaFold insere-se na **biologia estrutural** e, no contexto das ómicas, na **proteómica estrutural**: faz a ponte entre a informação de sequência produzida pela genómica e pela transcriptómica e a estrutura tridimensional, que determina a função. Ao prever estruturas à escala de proteomas completos, tornou-se também uma ferramenta de **genómica estrutural**: a primeira aplicação ao proteoma humano cobriu 98,5% das proteínas, com 58% dos resíduos previstos com confiança (pLDDT > 70) \cite{tunyasuvunakool2021}. Com o AlphaFold-Multimer \cite{evans2021multimer} e sobretudo com o AF3, que modela complexos de proteínas, ácidos nucleicos e pequenas moléculas, estende-se ainda à **interactómica estrutural**.

## Dados de entrada e tecnologias associadas

Ao contrário da maioria das ferramentas ómicas, o AlphaFold não processa dados brutos de sequenciação ou de espetrometria de massa. O input do AF2 é apenas a **sequência de aminoácidos** de uma ou mais cadeias, em formato **FASTA**. O AF3 aceita um ficheiro **JSON** que descreve todas as entidades do sistema: cadeias de proteína, DNA e RNA, ligandos (códigos do *Chemical Component Dictionary* do PDB ou SMILES), iões e modificações pós-traducionais ou de nucleótidos \cite{af3github}. Durante a previsão, a ferramenta consulta bases de dados de sequências (UniRef, MGnify, BFD) e de estruturas (PDB).

As tecnologias associadas situam-se, portanto, a montante e a jusante. As sequências de entrada resultam tipicamente da tradução de genes identificados por sequenciação de genomas ou transcriptomas (Illumina, PacBio, Oxford Nanopore) ou de proteínas identificadas por proteómica baseada em espetrometria de massa (LC-MS/MS). Por outro lado, o modelo foi treinado com estruturas determinadas experimentalmente e depositadas no PDB, por **cristalografia de raios-X**, **criomicroscopia eletrónica** (crio-EM) e **ressonância magnética nuclear** (RMN), e é com estas técnicas que os seus resultados são validados.

## Pergunta biológica

A pergunta central é: **qual é a estrutura tridimensional de uma proteína, conhecida apenas a sua sequência?** No AF3, acrescenta-se: **como interage com outras moléculas?** A resposta tem aplicações diretas na anotação funcional de proteínas não caracterizadas, na interpretação do impacto de variantes genéticas (o AlphaMissense, derivado do AlphaFold, classifica variantes *missense* à escala do proteoma \cite{cheng2023alphamissense}), no apoio à determinação experimental de estruturas (substituição molecular em cristalografia, interpretação de mapas de crio-EM), no desenho racional de fármacos e no desenho de proteínas.

# Arquitetura e Implementação Computacional

## Linguagens e bibliotecas

Ambas as versões estão escritas em **Python** e assentam na biblioteca de diferenciação automática **JAX**, com a camada de redes neuronais **Haiku** (ambas da Google). No AlphaFold 2 (AF2), a construção dos inputs recorre a ferramentas externas consagradas: **HMMER** (*jackhmmer*) e **HH-suite** (*HHblits*, *HHsearch*) para as pesquisas de homólogos e de *templates*, **Kalign** para alinhamentos e **OpenMM** para a relaxação final com o campo de forças Amber \cite{jumper2021,af2github}. O AlphaFold 3 (AF3) acrescenta componentes em **C++** (expostos ao Python via *pybind11*) para o processamento de dados e **RDKit** para a química de ligandos \cite{af3github}.

## Formas de acesso e instalação

Há quatro formas de usar o AlphaFold, que diferem bastante nos requisitos de hardware e no grau de controlo sobre a previsão (Tabela \ref{tab:acesso}). Neste trabalho usaram-se as três primeiras, cujos resultados são apresentados na Secção 4.3.

\begingroup\small

| Via | Versão | Requisitos | Interface | Principais limitações |
|:-------------|:--------|:--------------------|:--------|:------------------------------|
| AlphaFold Server | AF3 | Browser e conta Google | Web | Quota diária; uso não comercial; ligandos e modificações restritos a uma lista |
| ColabFold | AF2 (+ Multimer) | Browser (GPU gratuita do Colab) ou instalação local | Notebook / CLI | Memória da GPU gratuita limita o tamanho; sessões com tempo limitado |
| AlphaFold DB | AF2 (pré-calculado) | Nenhum | Web / API REST | Só cadeias proteicas naturais; sem complexos, ligandos ou mutantes |
| Instalação local | AF2 e AF3 | Linux, GPU NVIDIA, 0,6--2,6 TB de bases de dados | CLI (Docker) | Hardware exigente; pesos do AF3 só mediante autorização |

: Formas de acesso ao AlphaFold. {#tab:acesso}

\endgroup

**AlphaFold Server.** Lançado em 2024 com o AF3 \cite{abramson2024af3}, é a forma mais simples de aceder à versão mais recente: no browser (\url{https://alphafoldserver.com}) introduzem-se as cadeias de proteína, DNA ou RNA, escolhem-se ligandos, iões e modificações pós-traducionais de uma lista predefinida, e em poucos minutos obtém-se um arquivo com as estruturas (mmCIF) e as métricas de confiança (pLDDT, PAE, pTM/ipTM). Em contrapartida, há um número limitado de previsões por dia, o uso está restrito a fins não comerciais e não é possível automatizar submissões nem usar ligandos arbitrários.

**ColabFold.** Desenvolvido pela comunidade académica, o ColabFold substitui as pesquisas de homólogos do AF2 (HMMER/HH-suite sobre bases de dados de vários TB) por um servidor público **MMseqs2**, tornando a construção do alinhamento múltiplo 40 a 60 vezes mais rápida sem perda relevante de exatidão \cite{mirdita2022colabfold}. Corre num *notebook* Google Colab com GPU gratuita, onde basta colar a sequência (cadeias de um complexo separadas por "`:`"), ou localmente através do pacote `colabfold`, disponível no **Bioconda** (v1.5.5) e no projeto *LocalColabFold*. É a forma mais prática de usar o AF2 sem infraestrutura própria e permite controlar parâmetros como o número de modelos, de ciclos de *recycling* ou o uso de *templates*.

**AlphaFold Protein Structure Database (AFDB).** Mantida pelo EMBL-EBI em parceria com a DeepMind, disponibiliza previsões AF2 já calculadas para mais de 214 milhões de sequências da UniProt \cite{varadi2022afdb,varadi2024afdb}, sob licença CC BY 4.0. Cada entrada (pesquisável por nome ou acesso UniProt) inclui o modelo 3D colorido por pLDDT, o mapa de PAE e ficheiros PDB/mmCIF; há ainda uma API REST pública e descarga em massa. Por exemplo, `https://alphafold.ebi.ac.uk/api/prediction/P04637` devolve as ligações para o modelo da p53 humana (versão 6 dos modelos). A AFDB não exige qualquer computação, mas limita-se a cadeias proteicas naturais isoladas.

**Instalação local.** O AF2 é instalado através de **Docker** (imagem construída a partir do repositório oficial) e requer Linux, uma GPU NVIDIA e cerca de 556 GB de bases de dados genéticas comprimidas (2,62 TB descomprimidas), existindo uma versão reduzida com menores requisitos de disco \cite{af2github}. O AF3 exige uma GPU com *compute capability* $\geq$ 8.0 (validado em A100 e H100 de 80 GB para até 5120 *tokens*), pelo menos 64 GB de RAM e cerca de 630 GB de bases de dados; a instalação é também por Docker, com instruções oficiais para conversão em **Singularity/Apptainer** para utilização em clusters \cite{af3github}. Nenhuma das versões oficiais está no Bioconda, embora todas as dependências externas (HMMER, HH-suite, Kalign, OpenMM) o estejam. Existem ainda portes para outro hardware, como a versão do AF2 otimizada pela Intel para CPUs Xeon \cite{intelopenomics}.

## Interface

O AlphaFold não tem uma interface gráfica local: as versões oficiais correm por **linha de comandos** (`run_alphafold.py`, com input FASTA no AF2 e JSON no AF3), o ColabFold funciona como *notebook* interativo ou CLI (`colabfold_batch`), e o AlphaFold Server e a AFDB são **aplicações web**. Para visualizar e analisar os resultados recorre-se a ferramentas externas como PyMOL, ChimeraX ou o visualizador Mol\* integrado nas páginas web.

In [ ]:
# === Caso de estudo: cálculo das métricas (não aparece no PDF) ===
import sys
sys.path.insert(0, str(ROOT / "scripts"))
from analise_af import *

def f(x, d=2):
    """Número com vírgula decimal (português)."""
    return "--" if x is None else f"{x:.{d}f}".replace(".", ",")

res = {}
# Ubiquitina: referência 1UBQ; AFDB só tem precursores -> P62987 (ubiquitina + eL40), resíduos 1-76
ubq_ref = load_pdb("1UBQ")
ubq = {"AF2": load_af2("ubiquitina"), "AF3": load_af3("ubiquitina"), "AFDB": load_afdb("P62987")}
for k, p in ubq.items():
    pl = [v for n, v in p.plddt("A").items() if n <= 76]
    res[("ubq", k)] = dict(
        plddt=np.mean(pl), ptm=p.ptm,
        rmsd=superpor(ubq_ref, p.st, [("A", "A")], residuos=set(range(1, 77))).rmsd,
        rmsd_nucleo=superpor(ubq_ref, p.st, [("A", "A")], residuos=set(range(1, 71))).rmsd,
        tm=tm_score(ubq_ref, "A", p.st, "A"))

# Hemoglobina: estado T = 2HHB (desoxi), estado R = 1HHO (oxi; tetrâmero da montagem biológica)
T, R = load_pdb("2HHB"), load_pdb("1HHO", assembly=True)
aT, bT = cadeias_por_inicio(T, "VLS"), cadeias_por_inicio(T, "VHL")
aR, bR = cadeias_por_inicio(R, "VLS"), cadeias_por_inicio(R, "VHL")
res[("hb", "T_vs_R")] = melhor_mapeamento(T, R, [(aT, aR), (bT, bR)]).rmsd
hb = {"AF2": load_af2("hemoglobina"), "AF3": load_af3("hemoglobina")}
for k, p in hb.items():
    a, b = cadeias_por_inicio(p.st, "VLS"), cadeias_por_inicio(p.st, "VHL")
    sT = melhor_mapeamento(T, p.st, [(aT, a), (bT, b)])
    sR = melhor_mapeamento(R, p.st, [(aR, a), (bR, b)])
    res[("hb", k)] = dict(plddt=p.plddt_medio(a + b), ptm=p.ptm, iptm=p.iptm, rmsd_T=sT.rmsd, rmsd_R=sR.rmsd,
                          rmsd_cadeia=np.mean([superpor(T, p.st, [(sT.mapa[c], c)]).rmsd for c in a + b]),
                          sup=sT)
res[("hb", "fe_his_ref")] = fe_his_proximal(T)
res[("hb", "fe_his_af3")] = fe_his_proximal(hb["AF3"].st)
res[("hb", "hemes_af3")] = desvio_hemes(T, hb["AF3"].st, res[("hb", "AF3")]["sup"])
# Cadeias isoladas da AFDB (numeração UniProt inclui a Met1 -> offset -1)
hb_afdb = {"alfa": (load_afdb("P69905"), "A"), "beta": (load_afdb("P68871"), "B")}
for k, (p, ch) in hb_afdb.items():
    res[("hb_afdb", k)] = dict(plddt=p.plddt_medio(),
                               rmsd=superpor(T, p.st, [(ch, "A")], offsets={"A": -1}).rmsd,
                               tm=tm_score(T, ch, p.st, "A"))
for k, v in res.items():
    print(k, {kk: vv for kk, vv in v.items() if kk != "sup"} if isinstance(v, dict) else v)

In [ ]:
# === Figura do fluxo de trabalho (figures/workflow.pdf) ===
import subprocess
subprocess.run([sys.executable, str(ROOT / "scripts" / "figura_workflow.py")], check=True)

In [ ]:
# === Figuras do caso de estudo (guardadas em figures/) ===
from matplotlib.colors import LinearSegmentedColormap
plt.rcParams.update({"font.family": "Arial", "font.size": 8, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": "#888888", "axes.labelcolor": "#333333",
                     "xtick.color": "#555555", "ytick.color": "#555555", "legend.frameon": False})
# Paleta categórica fixa (AF2, AF3, AFDB) + estilo de linha como codificação secundária
COR = {"AF2": "#2a78d6", "AF3": "#eb6834", "AFDB": "#1baf7a"}
LS = {"AF2": "-", "AF3": "--", "AFDB": ":"}

def faixas_plddt(ax):
    for y in (50, 70, 90):
        ax.axhline(y, color="#bbbbbb", lw=0.6, zorder=0)
    ax.set_ylim(20, 101)
    ax.set_ylabel("pLDDT")

fig, axs = plt.subplots(1, 3, figsize=(6.9, 2.1), gridspec_kw={"width_ratios": [76, 141, 146]}, sharey=True)
paineis = [("Ubiquitina", {k: (p, "A", 0) for k, p in ubq.items()}, 76),
           ("Hemoglobina, cadeia α", {"AF2": (hb["AF2"], "A", 0), "AF3": (hb["AF3"], "A", 0),
                                      "AFDB": (hb_afdb["alfa"][0], "A", -1)}, 141),
           ("Hemoglobina, cadeia β", {"AF2": (hb["AF2"], "C", 0), "AF3": (hb["AF3"], "C", 0),
                                      "AFDB": (hb_afdb["beta"][0], "A", -1)}, 146)]
for ax, (titulo, series, n) in zip(axs, paineis):
    faixas_plddt(ax)
    for k, (p, ch, off) in series.items():
        d = {r + off: v for r, v in p.plddt(ch).items() if 1 <= r + off <= n}
        ax.plot(list(d), list(d.values()), color=COR[k], ls=LS[k], lw=1.3, label=k)
    ax.set_title(titulo, fontsize=8, loc="left", color="#333333")
    ax.set_xlim(1, n)
    ax.set_xlabel("Resíduo")
axs[0].legend(loc="lower left", handlelength=2.2)
for y, t in [(95, "muito alta"), (80, "confiante"), (60, "baixa"), (35, "muito baixa")]:
    axs[2].text(147.5, y, t, fontsize=6.5, color="#777777", va="center", clip_on=False)
fig.tight_layout(w_pad=0.6)
fig.savefig(FIG / "plddt.pdf")

# PAE da hemoglobina (só tokens proteicos): AF2-Multimer vs AF3
azuis = LinearSegmentedColormap.from_list("azuis", ["#0d366b", "#2a78d6", "#86b6ef", "#e8f1fc"])
tok = np.array(hb["AF3"].meta["token_chain_ids"])
prot = np.isin(tok, ["A", "B", "C", "D"])
paes = {"AF2-Multimer (ColabFold)": hb["AF2"].pae, "AF3 (AlphaFold Server)": hb["AF3"].pae[np.ix_(prot, prot)]}
fig, axs = plt.subplots(1, 2, figsize=(5.2, 2.5))
lim = np.cumsum([0, 141, 141, 146, 146])
for ax, (titulo, m) in zip(axs, paes.items()):
    im = ax.imshow(m, cmap=azuis, vmin=0, vmax=30, interpolation="nearest")
    for x in lim[1:-1]:
        ax.axhline(x - 0.5, color="white", lw=0.8); ax.axvline(x - 0.5, color="white", lw=0.8)
    centros = (lim[:-1] + lim[1:]) / 2
    nomes = ["α1", "α2", "β1", "β2"]
    ax.set_xticks(centros, nomes); ax.set_yticks(centros, nomes)
    ax.tick_params(length=0)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title(titulo, fontsize=8, color="#333333")
cb = fig.colorbar(im, ax=axs, shrink=0.8, pad=0.03)
cb.set_label("PAE (Å)"); cb.outline.set_visible(False)
fig.savefig(FIG / "pae_hemoglobina.pdf")
plt.close("all")

In [ ]:
# === Tabelas do caso de estudo (geradas em figures/*.tex e incluídas com \input) ===
linhas = []
for k in ["AF2", "AF3", "AFDB"]:
    r = res[("ubq", k)]
    linhas.append(f"Ubiquitina & {k} & {f(r['plddt'],1)} & {f(r['ptm'])} & {f(r['rmsd'])} & {f(r['rmsd_nucleo'])} & {f(r['tm'],3)} \\\\")
linhas.append(r"\midrule")
for k, nome in [("alfa", "Hb α (isolada)"), ("beta", "Hb β (isolada)")]:
    r = res[("hb_afdb", k)]
    linhas.append(f"{nome} & AFDB & {f(r['plddt'],1)} & -- & {f(r['rmsd'])} & -- & {f(r['tm'],3)} \\\\")
(FIG / "tab_monomeros.tex").write_text(r"""\begin{table}[H]\centering\small
\caption{Previsões de cadeias isoladas comparadas com as estruturas experimentais (1UBQ; 2HHB). RMSD dos Cα após sobreposição; o \emph{núcleo} da ubiquitina exclui a cauda C-terminal flexível (resíduos 71--76). TM-score normalizado pelo comprimento da estrutura experimental.}\label{tab:monomeros}
\begin{tabular}{llccccc}\toprule
Proteína & Fonte & pLDDT & pTM & RMSD (Å) & RMSD núcleo (Å) & TM-score \\\midrule
""" + "\n".join(linhas) + "\n\\bottomrule\\end{tabular}\\end{table}\n")

linhas = []
for k, nome in [("AF2", "AF2-Multimer"), ("AF3", "AF3 (+4 HEM)")]:
    r = res[("hb", k)]
    linhas.append(f"{nome} & {f(r['plddt'],1)} & {f(r['iptm'])} & {f(r['rmsd_cadeia'])} & {f(r['rmsd_T'])} & {f(r['rmsd_R'])} \\\\")
linhas.append(r"\midrule")
linhas.append(f"1HHO (estado R, experimental) & -- & -- & -- & {f(res[('hb','T_vs_R')])} & 0,00 \\\\")
(FIG / "tab_hemoglobina.tex").write_text(r"""\begin{table}[H]\centering\small
\caption{Estrutura quaternária prevista para a hemoglobina ($\alpha_2\beta_2$). RMSD dos Cα (574 resíduos) após sobreposição global ao estado T (2HHB, desoxi) e ao estado R (1HHO, oxi); \emph{RMSD cadeia} é a média das quatro cadeias sobrepostas individualmente ao estado T. A última linha dá a diferença experimental entre os dois estados como escala de referência.}\label{tab:hemoglobina}
\begin{tabular}{lccccc}\toprule
Modelo & pLDDT & ipTM & RMSD cadeia (Å) & RMSD vs T (Å) & RMSD vs R (Å) \\\midrule
""" + "\n".join(linhas) + "\n\\bottomrule\\end{tabular}\\end{table}\n")

# Algoritmo e Fluxo de Trabalho

## Princípio algorítmico

O AlphaFold é uma **rede neuronal profunda** treinada de ponta a ponta para converter uma sequência, e os seus homólogos, em coordenadas atómicas. O princípio biológico explorado é a **coevolução**: resíduos em contacto na estrutura tendem a mutar de forma correlacionada ao longo da evolução, deixando assinaturas no alinhamento múltiplo (MSA) que a rede aprende a interpretar geometricamente.

**AlphaFold 2.** A rede mantém duas representações: uma do MSA (sequências × resíduos) e uma de **pares de resíduos** (resíduos × resíduos), que codifica as relações espaciais. O tronco, o **Evoformer** (48 blocos), troca informação entre ambas através de mecanismos de atenção; as **atualizações triangulares** impõem consistência geométrica (as distâncias *i–j* e *j–k* restringem a distância *i–k*). O **Structure Module** trata cada resíduo como um corpo rígido (rotação + translação) e refina as suas posições com *Invariant Point Attention*, uma atenção invariante a rotações e translações globais, prevendo também os ângulos de torção das cadeias laterais. O processo é repetido três vezes (*recycling*), reutilizando as previsões anteriores como input, e uma relaxação com o campo de forças Amber remove choques estereoquímicos. O treino usou estruturas do PDB e uma etapa de *self-distillation* com estruturas previstas pelo próprio modelo, minimizando a FAPE (*frame aligned point error*); as métricas de confiança são previstas por "cabeças" adicionais da rede \cite{jumper2021}. O AlphaFold-Multimer reutiliza esta arquitetura, treinada com complexos e com MSA emparelhados por espécie \cite{evans2021multimer}.

**AlphaFold 3.** O AF3 generaliza o problema a qualquer biomolécula: proteínas e ácidos nucleicos são representados por um *token* por resíduo ou nucleótido, e ligandos e resíduos modificados por um *token* por átomo. O processamento do MSA foi reduzido a um pequeno módulo (4 blocos) e o Evoformer substituído pelo **Pairformer** (48 blocos), que opera apenas nas representações de pares e de *tokens*. A principal inovação é o **módulo de difusão**, que substitui o Structure Module: tal como os modelos generativos de imagem, parte de coordenadas aleatórias e remove "ruído" iterativamente até obter as posições de todos os átomos, dispensando a parametrização por ângulos de torção e as penalizações estereoquímicas do AF2. Por ser generativo, produz várias amostras por previsão, mas pode também "alucinar" estrutura em regiões desordenadas, problema mitigado incluindo no treino previsões do AF2 (*cross-distillation*) \cite{abramson2024af3}.

## Fluxo de trabalho

A Figura \ref{fig:workflow} resume o fluxo de trabalho das duas versões. A pesquisa de homólogos (passos 1--2) corre em CPU e é a etapa mais demorada numa instalação local; a inferência da rede (passos 3--4) corre em GPU.

![Fluxo de trabalho do AlphaFold 2 (azul) e do AlphaFold 3 (laranja). Os passos 1, 2 e 5 são comuns às duas versões, com as diferenças indicadas.](figures/workflow.pdf){#fig:workflow width=100%}

Em todas as vias, o resultado de uma previsão é um conjunto de ficheiros com a mesma lógica (verificado nas previsões feitas neste trabalho):

* **Estruturas 3D** -- cinco modelos ordenados por confiança: ficheiros PDB `*_rank_001` a `*_rank_005` (ColabFold) ou mmCIF `*_model_0` a `*_model_4` (AF3). O pLDDT de cada resíduo/átomo fica guardado na coluna *B-factor*, pelo que qualquer visualizador pode colorir o modelo pela confiança.
* **Métricas de confiança** -- ficheiros JSON com o pLDDT, a matriz de PAE, o pTM e (em complexos) o ipTM; no AF3, um ficheiro-resumo por modelo inclui ainda a fração desordenada e a deteção de choques estereoquímicos.
* **Inputs intermédios** -- o alinhamento múltiplo (formato A3M) e, quando usados, os *templates* do PDB, permitindo verificar a profundidade do MSA, um fator determinante da exatidão.
* **Registo e reprodutibilidade** -- parâmetros da execução (`config.json` no ColabFold; pedido em JSON e *seed* no AF3), registo da execução e, no ColabFold, gráficos prontos de pLDDT, PAE e cobertura do MSA.

## Validação e interpretação dos resultados

O AlphaFold acompanha cada previsão de estimativas da sua própria confiança, aprendidas durante o treino \cite{jumper2021}:

* **pLDDT** (*predicted local distance difference test*, 0--100), por resíduo: estimativa do lDDT-Cα \cite{mariani2013lddt} que o modelo obteria face à estrutura real. Por convenção: > 90, confiança muito alta (cadeias laterais geralmente corretas); 70--90, cadeia principal correta; 50--70, baixa; < 50, muito baixa, frequentemente indicando regiões intrinsecamente desordenadas. É armazenado na coluna *B-factor* dos ficheiros PDB/mmCIF.
* **PAE** (*predicted aligned error*, em Å), matriz $N \times N$: erro esperado na posição do resíduo $x$ quando a estrutura prevista e a real são alinhadas pelo resíduo $y$. Avalia a confiança na **posição relativa** de domínios e cadeias, algo que o pLDDT, sendo local, não mede.
* **pTM** e **ipTM** (0--1): estimativas do TM-score da estrutura global e das interfaces entre cadeias. Um pTM > 0,5 indica que a topologia global está provavelmente correta; para complexos, um ipTM > 0,8 corresponde a previsões de confiança elevada e < 0,6 sugere uma previsão falhada.

Estas métricas são calibradas dentro de cada modelo e **não devem ser comparadas diretamente entre versões** (por exemplo, um ipTM mais alto no AF2-Multimer do que no AF3 não significa um modelo melhor). A validação objetiva exige comparação com estruturas experimentais, usando o **RMSD** (desvio quadrático médio após sobreposição ótima, em Å; sensível a regiões flexíveis), o **TM-score** (0--1, independente do comprimento; > 0,5 indica o mesmo enovelamento) ou o **lDDT** (que dispensa sobreposição). 

**Exemplo prático.** Para demonstrar o uso das três vias de acesso e o tipo de resultado que devolvem, previu-se a estrutura de duas proteínas humanas: a **ubiquitina** (76 resíduos) e o tetrâmero da **hemoglobina** ($\alpha_2\beta_2$, com quatro grupos heme). O AF2 foi corrido no ColabFold v1.6.3 (MSA por MMseqs2, sem *templates*), o AF3 no AlphaFold Server (configuração por omissão, com *templates*; quatro ligandos HEM no caso da hemoglobina) e da AFDB descarregaram-se, via API, os modelos das cadeias α (P69905) e β (P68871) e, para a ubiquitina, o precursor P62987 (resíduos 1--76), já que na UniProt a ubiquitina madura não existe como entrada própria. As previsões foram comparadas com estruturas experimentais (PDB 1UBQ, 1,8 Å; 2HHB, 1,74 Å) usando código próprio em Python (*gemmi*; sobreposição de Kabsch; TM-align).

O pLDDT (Figura \ref{fig:plddt}) é superior a 90 em quase todos os resíduos e só desce na cauda C-terminal da ubiquitina (LRLRGG), uma região flexível, que é a que estabelece a ligação às proteínas-alvo. A comparação com as estruturas experimentais confirma esta leitura (Tabela \ref{tab:monomeros}): TM-score de 0,96 e RMSD do núcleo inferior a 0,5 Å, abaixo da própria resolução cristalográfica. As previsões do ColabFold e da AFDB (AF2 oficial, com HMMER/HH-suite) são praticamente idênticas, o que confirma que a pesquisa de homólogos com MMseqs2 não tem custo de exatidão \cite{mirdita2022colabfold}.

\input{../figures/tab_monomeros.tex}

![Confiança por resíduo (pLDDT) das previsões AF2 (ColabFold), AF3 (AlphaFold Server) e AlphaFold DB. As linhas horizontais marcam os limiares 50, 70 e 90 das categorias de confiança.](figures/plddt.pdf){#fig:plddt width=100%}

No complexo, o PAE (Figura \ref{fig:pae}) é baixo não só dentro de cada cadeia mas também entre cadeias, indicando que a posição relativa das quatro subunidades é bem determinada (ipTM de 0,93 no AF2-Multimer e 0,88 no AF3). O AF3 colocou ainda os quatro hemes nos bolsos corretos: após sobreposição da proteína, os átomos de ferro distam 0,4--0,8 Å das posições cristalográficas, coordenados pela histidina proximal correta (His87 nas cadeias α, His92 nas β).

![Erro de alinhamento previsto (PAE) para o tetrâmero da hemoglobina. Cada bloco corresponde a um par de cadeias; tons escuros indicam baixo erro esperado na posição relativa.](figures/pae_hemoglobina.pdf){#fig:pae width=68%}

# Reprodutibilidade, Comunidade e Limitações

## Licenciamento

O AlphaFold é **parcialmente open-source**. No AF2, tanto o código (licença Apache 2.0) como os parâmetros treinados (CC BY 4.0) são abertos e permitem uso comercial \cite{af2github}. No AF3, o código está atualmente disponível sob Apache 2.0, mas os **pesos do modelo** só podem ser usados se obtidos diretamente da Google, mediante pedido e sob termos próprios que restringem o uso a fins não comerciais; o AlphaFold Server tem restrições semelhantes, e o uso comercial é oferecido através da Google Cloud \cite{af3github}. As previsões da AFDB são distribuídas sob CC BY 4.0 e o ColabFold sob licença MIT. Estas restrições motivaram reimplementações abertas como o OpenFold \cite{ahdritz2024openfold} e o OpenFold3 \cite{openfold3github} (Apache 2.0), o Boltz (MIT) \cite{wohlwend2024boltz} e o Chai-1 \cite{chai2024}.

## Comunidade e manutenção

A comunidade é muito ativa (Tabela \ref{tab:github}). O repositório do AF2 está essencialmente em modo de manutenção, sem novas versões desde abril de 2023, embora continue a ser o mais popular. O AF3 recebe versões regulares (três entre junho e outubro de 2026) e tem poucos problemas (*issues*) em aberto, sinal de manutenção ativa. O ColabFold, mantido pela comunidade académica, lança versões a cada poucos meses, e a AFDB foi atualizada para a versão 6 dos modelos em 2025. A documentação é extensa: READMEs detalhados, um curso online da EMBL-EBI e páginas de ajuda do AlphaFold Server. O impacto científico é excecional: segundo a base de dados OpenAlex, o artigo do AF2 tem mais de 45 000 citações e o do AF3 cerca de 18 000 (novembro de 2026).

\begingroup\small

| Repositório | Estrelas | Última versão | Licença (código) |
|:------------------------------|----------:|:----------------------|:----------|
| google-deepmind/alphafold (AF2) | 14 939 | v2.3.2 (abr. 2023) | Apache 2.0 |
| google-deepmind/alphafold3 (AF3) | 8 772 | v3.0.5 (out. 2026) | Apache 2.0 |
| sokrypton/ColabFold | 2 989 | v1.6.3 (set. 2026) | MIT |
| jwohlwend/boltz | 4 322 | v2.2.1 (set. 2025) | MIT |
| aqlaboratory/openfold-3 | 919 | 0.5.1 (out. 2026) | Apache 2.0 |

: Atividade dos repositórios do AlphaFold e de alternativas abertas (GitHub, 13/11/2026). {#tab:github}

\endgroup

## Limitações

* **Uma única conformação estática.** O AlphaFold devolve uma estrutura, enquanto muitas proteínas funcionam alternando entre várias. O nosso exemplo ilustra-o bem (Tabela \ref{tab:hemoglobina}): com cadeias individuais quase idênticas (RMSD $\leq$ 0,6 Å), o AF3 previu o tetrâmero da hemoglobina no **estado T** (desoxi, 2HHB; 0,43 Å), enquanto o AF2-Multimer ficou mais próximo do **estado R** (oxi, 1HHO; 1,01 Å), ambos com confiança muito alta. A diferença pode dever-se aos hemes ou aos *templates*, só usados no AF3, mas a lição é a mesma: a confiança elevada não indica qual estado fisiológico foi previsto, e a transição T$\leftrightarrow$R, base da ligação cooperativa do oxigénio, está fora do alcance do modelo.
* **Regiões intrinsecamente desordenadas.** Recebem pLDDT baixo (< 50) e aparecem como "fitas" sem estrutura. O AF3, por ser um modelo generativo, pode ainda "alucinar" estrutura ordenada nessas regiões, um problema reconhecido pelos autores \cite{abramson2024af3}.
* **Efeito de mutações pontuais.** O AlphaFold não foi treinado para prever o impacto estrutural de mutações *missense* e tende a devolver a mesma estrutura para a proteína selvagem e mutante, mesmo em mutações que a desestabilizam \cite{buel2022mutations}.
* **Dependência do alinhamento múltiplo.** A exatidão do AF2 desce de forma acentuada quando o MSA tem menos de cerca de 30 sequências efetivas \cite{jumper2021}, o que afeta proteínas "órfãs", sem homólogos conhecidos.
* **Ligandos e complexos.** O AF2 não modela ligandos, iões nem modificações; o AF3 modela-os, mas no AlphaFold Server apenas a partir de uma lista fechada. A AFDB contém só cadeias isoladas tal como codificadas nos genes: não inclui complexos nem formas maduras (a ubiquitina, por exemplo, só existe como precursor).
* **Recursos e licenças.** A instalação local exige GPU e centenas de GB a vários TB de disco (Secção 3.2), e a utilização dos pesos do AF3 e do AlphaFold Server está restrita a fins não comerciais.

\input{../figures/tab_hemoglobina.tex}

# Conclusão

O AlphaFold transformou a biologia estrutural: um problema em aberto durante cinco décadas passou a ter, para a maioria das proteínas, uma solução computacional com exatidão próxima da experimental, acessível a qualquer investigador através do AlphaFold Server, do ColabFold ou da AlphaFold DB. O AF2 consolidou-se como uma ferramenta madura e totalmente aberta para proteínas; o AF3 alargou o alcance a complexos com ácidos nucleicos, ligandos e modificações, à custa de uma licença mais restritiva. Os nossos exemplos confirmam os dois lados: previsões com desvios inferiores a 0,5 Å face às estruturas cristalográficas e grupos heme corretamente posicionados, mas também duas conformações quaternárias diferentes da mesma proteína, ambas com confiança muito alta.

A principal lição é que o AlphaFold produz **hipóteses estruturais de alta qualidade**, que devem ser interpretadas à luz das métricas de confiança e confirmadas experimentalmente quando a questão biológica depende de dinâmica conformacional, ligandos ou mutações. O aparecimento de alternativas abertas (OpenFold3, Boltz) e de extensões como o AlphaMissense indica que esta área continuará a evoluir rapidamente.